# Step 6: Evaluation - baseline vs improved
Both `best.pt` checkpoints are evaluated on the **same** validation and test splits. Metrics: precision, recall, F1, mAP@50, mAP@50-95, parameters, GFLOPs, inference time.

Choose settings on the validation split; report the **test** split once at the end.

In [ ]:
# Set True for a fast CPU sanity run (3 epochs, 320 px, 25% of train). False = full experiment.
QUICK_TEST = False
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # works from VS Code either way
sys.path.insert(0, str(ROOT / "src"))
import mri_detection as md
from mri_detection import data, viz, models, train, evaluate
cfg = md.load_config(quick=QUICK_TEST)
RUNS = md.runs_dir(cfg)
YOLO_DIR = md.ROOT / cfg["dataset"]["yolo_dir"]
DATA_YAML = data.training_yaml(cfg, YOLO_DIR)
import yaml, random, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage
from ultralytics import YOLO
names = list(yaml.safe_load(open(DATA_YAML))["names"].values())
BEST_BASE = RUNS / "baseline_yolov8m" / "weights" / "best.pt"
BEST_PROP = RUNS / "proposed_yolov8m" / "weights" / "best.pt"
assert BEST_BASE.exists() and BEST_PROP.exists(), "Train both models first (notebooks 03 and 05)"
IMGSZ, BATCH = cfg["train"]["imgsz"], cfg["train"]["batch"]
DEVICE = md.config.resolve_device(cfg["train"]["device"])

## 6.1 Overall results

In [ ]:
results = {}
for split in ["val", "test"]:
    b_sum, b_cls = evaluate.evaluate(BEST_BASE, "baseline", DATA_YAML, RUNS, IMGSZ, BATCH, split, DEVICE)
    p_sum, p_cls = evaluate.evaluate(BEST_PROP, "proposed", DATA_YAML, RUNS, IMGSZ, BATCH, split, DEVICE)
    results[split] = (b_cls, p_cls)
    tab = pd.DataFrame({"Baseline YOLOv8m": b_sum, "Improved YOLOv8m": p_sum})
    tab["Change"] = tab["Improved YOLOv8m"] - tab["Baseline YOLOv8m"]
    print(f"\n===== {split.upper()} split =====")
    display(tab.round(4))
    tab.round(4).to_csv(RUNS / f"comparison_{split}.csv")

## 6.2 Per-class results

In [ ]:
for split, (b_cls, p_cls) in results.items():
    pc = b_cls.merge(p_cls, on="Class", suffixes=(" (base)", " (prop)"))
    for k in ["P", "R", "F1", "mAP@50", "mAP@50-95"]:
        pc[f"d {k}"] = pc[f"{k} (prop)"] - pc[f"{k} (base)"]
    print(f"\n===== per-class, {split} =====")
    display(pc.round(4))
    pc.round(4).to_csv(RUNS / f"per_class_{split}.csv", index=False)

## 6.3 Training curves

In [ ]:
cb, cp = evaluate.load_curve(RUNS, "baseline_yolov8m"), evaluate.load_curve(RUNS, "proposed_yolov8m")
panels = [("train/box_loss", "Train box loss"), ("val/box_loss", "Val box loss"), ("metrics/mAP50(B)", "Val mAP@50"),
          ("metrics/mAP50-95(B)", "Val mAP@50-95"), ("metrics/precision(B)", "Val precision"), ("metrics/recall(B)", "Val recall")]
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
for a, (col, t) in zip(ax.ravel(), panels):
    a.plot(cb["epoch"], cb[col], label="Baseline"); a.plot(cp["epoch"], cp[col], label="Improved")
    a.set_title(t); a.set_xlabel("epoch"); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.savefig(RUNS / "training_curves_comparison.png", dpi=150); plt.show()

## 6.4 Confusion matrices (validation, normalised)

In [ ]:
for n in ["baseline", "proposed"]:
    f = RUNS / f"{n}_val" / "confusion_matrix_normalized.png"
    print(n); display(IPImage(str(f), width=520)) if f.exists() else print("not found:", f)

## 6.5 Qualitative comparison: ground truth | baseline | improved

In [ ]:
val_imgs = sorted((YOLO_DIR / "val" / "images").iterdir())
random.seed(1); picks = random.sample(val_imgs, min(6, len(val_imgs)))
mb, mp_ = YOLO(str(BEST_BASE)), YOLO(str(BEST_PROP))
fig, ax = plt.subplots(len(picks), 3, figsize=(12, 4 * len(picks)), squeeze=False)
for i, p in enumerate(picks):
    gt = viz.draw_boxes(p, viz.read_yolo_labels(YOLO_DIR / "val" / "labels" / (p.stem + ".txt")), names)
    pb = mb.predict(str(p), imgsz=IMGSZ, conf=0.25, verbose=False, device=DEVICE)[0].plot()[..., ::-1]
    pp = mp_.predict(str(p), imgsz=IMGSZ, conf=0.25, verbose=False, device=DEVICE)[0].plot()[..., ::-1]
    for j, (im, t) in enumerate(zip([gt, pb, pp], ["Ground truth", "Baseline", "Improved"])):
        ax[i, j].imshow(im); ax[i, j].set_title(t); ax[i, j].axis("off")
plt.tight_layout(); plt.savefig(RUNS / "qualitative_comparison.png", dpi=130); plt.show()

## 6.6 Conclusion (fill in after the full run)
- **Overall:** compare mAP@50-95 and recall first; do not claim an improvement the tables do not show.
- **Per class:** which class gained or lost most?
- **Errors:** from the confusion matrices and the examples above.
- **Limits:** one dataset, one seed, one image-level split (no patient ids), so slices from one patient may appear in several splits. Repeat with at least three seeds and report mean +/- std. Research prototype, **not** a clinical tool.